# §11 体验实验：故障曲线诊断

预置笔记本。三个读数先回扣正文：批大小（噪声旋钮）、η（稳定性边缘位置 2/η）、锐度（Hessian 最大特征值）。四张训练卡只有标签（批大小, η）与损失曲线，请先写下你的诊断，再跑下面的格子核对。

| 卡 | 标签 (批, η) |
| --- | --- |
| 卡一 | (16, 0.8) |
| 卡二 | (16, 0.02) |
| 卡三 | (4, 0.5) |
| 卡四 | (16, 0.5) |

In [ ]:
import numpy as np
from pathlib import Path

HERE = Path('.')  # 本笔记本与 trace_*.tsv 同目录（assets/§11-体验实验/）
DIVERGE_LIMIT = 1e6

def load(name):
    rows = np.genfromtxt(HERE / f'trace_{name}.tsv', delimiter='	', skip_header=1)
    return rows  # step, loss, sharpness(nan=未测), val_loss

cards = ['lr_too_big', 'lr_too_small', 'small_batch', 'healthy']
for c in cards:
    r = load(c)
    over = r[r[:, 1] > DIVERGE_LIMIT]
    sharp = r[~np.isnan(r[:, 2])]
    tag = f'第 {int(over[0,0])} 步越过 1e6，发散' if len(over) else f'跑完 {int(r[-1,0])} 步'
    print(f'{c:14s} {tag}；末态锐度 {sharp[-1,2]:.2f}，末段验证损失 {r[-100:,3].mean():.3f}')

卡一 2/η = 2.5 低于初始锐度 3.88，每个不稳定方向都在放大误差——学习率太大。
卡二 2/η = 100，安全但极慢，600 步只挪动 0.024——蜗牛曲线。
卡三 批 4 的噪声把解推向平坦处，末态锐度远低于 2/η = 4。
卡四 锐度贴着 4 震荡——健康的边缘稳定。

延伸一：批大小扫描（η 都取 0.5，各 600 步；16 = 全批量）。末态锐度单调吗？末段验证损失与锐度同向吗？

In [ ]:
for b in (4, 8, 16):
    r = load(f'batch{b}')
    sharp = r[~np.isnan(r[:, 2])]
    print(f'批 {b:2d}：末态锐度 {sharp[-1,2]:.2f}，末段验证损失 {r[-100:,3].mean():.3f}')

延伸二：线性缩放对照。批 4／η 0.125／2400 步 vs 批 16／η 0.5／600 步，看过的样本数同为 9600。两条曲线大体同行吗？再想想：批从 4 提到 16，线性缩放律要求 η 到多少？这个建议值在本玩具里会发生什么？

In [ ]:
r4 = load('scaling_b4'); r16 = load('healthy')
print(f'批 4 组终点训练损失 {r4[-1,1]:.3f}（2400 步）；批 16 组终点 {r16[-1,1]:.3f}（600 步）')
rc = load('scaling_capped')
print(f"线性缩放建议值 η=2.0：第 {int(rc[-1,0])} 步 loss 达 {rc[-1,1]:.3g}，2/η={2/2.0:.0f}" )